# Machine learning · Pipelines y preprocesado

Un modelo real casi nunca recibe una matriz limpia de números: hay huecos, columnas de texto y escalas distintas. Este notebook practica cómo encadenar el preprocesado y el modelo con `Pipeline` y `ColumnTransformer` de scikit-learn, por qué eso evita la fuga de datos, cómo buscar hiperparámetros sobre el pipeline entero, cómo escribir transformadores propios, cómo seleccionar variables y cómo guardar el modelo con `joblib`. Los datos son alquileres vacacionales de Canarias generados con una semilla fija.

## 1. Alquileres vacacionales

Generar con una semilla 600 alquileres vacacionales con isla, tipo de alojamiento, superficie, habitaciones, distancia a la playa, piscina y valoración, y un precio por noche que dependa de ellas con algo de ruido. Dejar huecos (`NaN`) en algunas columnas, como en unos datos reales, y mostrar cuántos faltan en cada una.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 600
islas = ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera", "El Hierro"]
extra_isla = dict(zip(islas, [25, 20, 30, 15, 5, 10, 0]))
tipos = ["apartamento", "casa", "villa"]
extra_tipo = {"apartamento": 0, "casa": 15, "villa": 60}

isla = rng.choice(islas, n, p=[0.3, 0.25, 0.15, 0.12, 0.08, 0.06, 0.04])
tipo = rng.choice(tipos, n, p=[0.55, 0.3, 0.15])
superficie = np.round(rng.lognormal(4.1, 0.4, n))
habitaciones = np.clip(np.round(superficie / 30 + rng.normal(0, 0.6, n)), 1, 6)
distancia = np.round(rng.exponential(2.5, n), 2)
piscina = rng.random(n) < 0.3
valoracion = np.round(np.clip(rng.normal(4.2, 0.4, n), 1, 5), 1)
precio = (30 + 0.8 * superficie + 12 * habitaciones - 6 * distancia + 20 * piscina + 15 * (valoracion - 4)
          + np.vectorize(extra_isla.get)(isla) + np.vectorize(extra_tipo.get)(tipo) + rng.normal(0, 12, n))

alquileres = pd.DataFrame({"isla": isla, "tipo": tipo, "superficie": superficie, "habitaciones": habitaciones,
                           "distancia_playa": distancia, "piscina": piscina.astype(int), "valoracion": valoracion,
                           "precio": precio.round(2)})
for columna, fraccion in [("superficie", 0.05), ("valoracion", 0.12), ("isla", 0.04)]:
    alquileres.loc[rng.random(n) < fraccion, columna] = np.nan

print(alquileres.head())
print("\nHuecos por columna:")
print(alquileres.isna().sum())
assert alquileres.shape == (n, 8) and alquileres["precio"].notna().all()

            isla         tipo  superficie  habitaciones  distancia_playa  \
0  Fuerteventura  apartamento        41.0           1.0             1.55   
1   Gran Canaria         casa        43.0           1.0             0.31   
2       La Palma        villa       161.0           5.0             0.63   
3      Lanzarote         casa       124.0           5.0             2.00   
4       Tenerife        villa        51.0           1.0             0.13   

   piscina  valoracion  precio  
0        0         4.1   75.28  
1        0         4.1  115.37  
2        0         4.4  302.08  
3        0         4.4  240.83  
4        0         3.9  170.99  

Huecos por columna:
isla               26
tipo                0
superficie         29
habitaciones        0
distancia_playa     0
piscina             0
valoracion         79
precio              0
dtype: int64


## 2. Imputar y escalar las columnas numéricas

Separar las variables del precio y dividir en entrenamiento y prueba. Construir un `Pipeline` que rellene los huecos numéricos con la mediana (`SimpleImputer`) y después estandarice (`StandardScaler`). Comprobar que, tras `fit_transform` en entrenamiento, no quedan huecos y cada columna tiene media 0 y desviación 1, y que las medianas aprendidas son las del conjunto de entrenamiento.

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numericas = ["superficie", "habitaciones", "distancia_playa", "valoracion"]
categoricas = ["isla", "tipo"]
binarias = ["piscina"]
X = alquileres.drop(columns="precio")
y = alquileres["precio"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=0)

prepro_num = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
Z = prepro_num.fit_transform(X_ent[numericas])
print("Medianas aprendidas:", dict(zip(numericas, prepro_num.named_steps["imputar"].statistics_.tolist())))
print("Medias:", Z.mean(axis=0).round(6))
print("Desviaciones:", Z.std(axis=0).round(6))
assert not np.isnan(Z).any()
assert np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.std(axis=0), 1)
assert np.allclose(prepro_num.named_steps["imputar"].statistics_, X_ent[numericas].median())

Medianas aprendidas: {'superficie': 58.0, 'habitaciones': 2.0, 'distancia_playa': 1.83, 'valoracion': 4.2}
Medias: [-0.  0.  0.  0.]
Desviaciones: [1. 1. 1. 1.]


## 3. Codificar las columnas categóricas

Las islas y los tipos son texto. Crear un pipeline categórico que rellene la isla que falta con el valor más frecuente y la convierta en columnas 0/1 con `OneHotEncoder(handle_unknown="ignore")`. Mostrar los nombres de las columnas generadas y comprobar qué ocurre con una isla que no aparecía en el entrenamiento (La Graciosa): debe codificarse como todo ceros en lugar de dar error.

In [3]:
from sklearn.preprocessing import OneHotEncoder

prepro_cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                       ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
C = prepro_cat.fit_transform(X_ent[categoricas])
nombres = prepro_cat.get_feature_names_out()
print(len(nombres), "columnas:", list(nombres))
assert C.shape[1] == len(islas) + len(tipos)
assert (C.sum(axis=1) == 2).all()

nueva = pd.DataFrame({"isla": ["La Graciosa"], "tipo": ["casa"]})
codificada = prepro_cat.transform(nueva)[0]
print("La Graciosa, casa:", dict(zip(nombres, codificada.astype(int).tolist())))
assert codificada[: len(islas)].sum() == 0 and codificada.sum() == 1

10 columnas: ['isla_El Hierro', 'isla_Fuerteventura', 'isla_Gran Canaria', 'isla_La Gomera', 'isla_La Palma', 'isla_Lanzarote', 'isla_Tenerife', 'tipo_apartamento', 'tipo_casa', 'tipo_villa']
La Graciosa, casa: {'isla_El Hierro': 0, 'isla_Fuerteventura': 0, 'isla_Gran Canaria': 0, 'isla_La Gomera': 0, 'isla_La Palma': 0, 'isla_Lanzarote': 0, 'isla_Tenerife': 0, 'tipo_apartamento': 0, 'tipo_casa': 1, 'tipo_villa': 0}


## 4. `ColumnTransformer`: cada columna con su tratamiento

Reunir los dos pipelines anteriores en un `ColumnTransformer` que aplique el numérico a las columnas numéricas, el categórico a las de texto y deje pasar la piscina sin cambios. Pedir la salida como `DataFrame` con `set_output(transform="pandas")` y comprobar el número de columnas y que no queda ningún hueco.

In [4]:
from sklearn.compose import ColumnTransformer


def crear_prepro():
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), numericas),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                          ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categoricas),
        ("bin", "passthrough", binarias),
    ])


prepro = crear_prepro().set_output(transform="pandas")
X_ent_t = prepro.fit_transform(X_ent)
print(X_ent_t.shape)
print(X_ent_t.iloc[:3, [0, 1, 4, 5, 14]].round(2))
assert X_ent_t.shape == (len(X_ent), len(numericas) + len(islas) + len(tipos) + len(binarias))
assert not X_ent_t.isna().any().any()

(480, 15)
     num__superficie  num__habitaciones  cat__isla_El Hierro  \
118            -1.36              -1.07                  0.0   
362             1.29               2.79                  0.0   
55              1.52               0.86                  0.0   

     cat__isla_Fuerteventura  bin__piscina  
118                      0.0             1  
362                      0.0             0  
55                       1.0             0  


## 5. Preprocesado y modelo en un solo objeto

Añadir una regresión `Ridge` al final del preprocesado para tener un único estimador que recibe el `DataFrame` crudo. Entrenarlo, medir el $R^2$ y el error absoluto medio en prueba y compararlo con un `DummyRegressor` que siempre predice la media. Mostrar los cinco coeficientes más grandes con el nombre de su variable.

In [5]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

modelo = Pipeline([("prepro", crear_prepro()), ("modelo", Ridge(alpha=1.0))])
modelo.fit(X_ent, y_ent)
pred = modelo.predict(X_pru)
base = DummyRegressor().fit(X_ent, y_ent).predict(X_pru)

print(f"{'':8}{'R²':>8}{'MAE (€)':>10}")
print(f"{'Ridge':8}{r2_score(y_pru, pred):8.3f}{mean_absolute_error(y_pru, pred):10.2f}")
print(f"{'Media':8}{r2_score(y_pru, base):8.3f}{mean_absolute_error(y_pru, base):10.2f}")

coef = pd.Series(modelo.named_steps["modelo"].coef_, index=modelo.named_steps["prepro"].get_feature_names_out())
print("\nCoeficientes más grandes:")
print(coef.abs().sort_values(ascending=False).head(5).index.map(lambda c: f"{c:28} {coef[c]:8.2f}").str.cat(sep="\n"))
assert r2_score(y_pru, pred) > 0.8 > r2_score(y_pru, base)

              R²   MAE (€)
Ridge      0.923      9.98
Media     -0.004     35.68

Coeficientes más grandes:
cat__tipo_villa                 35.54
cat__tipo_apartamento          -25.13
num__superficie                 19.99
bin__piscina                    18.37
num__habitaciones               15.24


## 6. Fuga de datos al escalar

Comparar dos maneras de validar un `KNeighborsRegressor` con validación cruzada de 5 particiones sobre las columnas numéricas: (a) imputar y escalar todo el conjunto antes de validar y (b) poner el imputador y el escalador dentro del pipeline, de modo que en cada partición solo aprendan de los datos de entrenamiento. Medir la diferencia entre las dos puntuaciones y explicar por qué la segunda es la honesta aunque aquí la diferencia sea pequeña.

In [6]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor

particiones = KFold(5, shuffle=True, random_state=0)
X_num = X[numericas]

X_todo = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]).fit_transform(X_num)
con_fuga = cross_val_score(KNeighborsRegressor(10), X_todo, y, cv=particiones, scoring="r2")

sin_fuga = cross_val_score(
    Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler()),
              ("knn", KNeighborsRegressor(10))]),
    X_num, y, cv=particiones, scoring="r2")

print(f"Escalado antes de dividir: R² = {con_fuga.mean():.4f}")
print(f"Escalado dentro del pipeline: R² = {sin_fuga.mean():.4f}")
print(f"Diferencia: {con_fuga.mean() - sin_fuga.mean():+.4f}")
assert abs(con_fuga.mean() - sin_fuga.mean()) < 0.05

Escalado antes de dividir: R² = 0.6172
Escalado dentro del pipeline: R² = 0.6169
Diferencia: +0.0003


Con 600 filas, la media y la desviación de todo el conjunto apenas difieren de las de cada partición, así que la fuga al escalar es pequeña. Aun así la validación (a) ha visto los datos de prueba al calcular el escalado, y con conjuntos pequeños o pasos que aprenden más (imputar con un modelo, seleccionar variables, reducir dimensión) el sesgo crece, como muestra el siguiente ejercicio.

## 7. Fuga de datos al seleccionar variables

Generar 100 muestras con 5000 variables de puro ruido y una etiqueta aleatoria, de modo que ningún modelo pueda acertar más del 50 %. Elegir las 20 variables más relacionadas con la etiqueta con `SelectKBest` usando todos los datos y validar después una regresión logística: la exactitud sale muy alta. Repetirlo con la selección dentro del pipeline y comprobar que vuelve al 50 %.

In [7]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

rng_ruido = np.random.default_rng(7)
X_ruido = rng_ruido.normal(size=(100, 5000))
y_ruido = rng_ruido.integers(0, 2, 100)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

X_elegidas = SelectKBest(f_classif, k=20).fit_transform(X_ruido, y_ruido)
trampa = cross_val_score(LogisticRegression(), X_elegidas, y_ruido, cv=cv).mean()
honesta = cross_val_score(Pipeline([("elegir", SelectKBest(f_classif, k=20)), ("modelo", LogisticRegression())]),
                          X_ruido, y_ruido, cv=cv).mean()
print(f"Selección antes de validar: {trampa:.2f}")
print(f"Selección dentro del pipeline: {honesta:.2f}")
assert trampa > 0.8 and abs(honesta - 0.5) < 0.2

Selección antes de validar: 0.83
Selección dentro del pipeline: 0.50


## 8. `GridSearchCV` sobre el pipeline

Buscar a la vez la estrategia de imputación numérica (media o mediana), si se escala o no y la regularización `alpha` de la regresión `Ridge`. Los parámetros de los pasos internos se nombran con dobles guiones bajos (`prepro__num__imputar__strategy`) y un paso entero se puede sustituir por `"passthrough"`. Mostrar las cinco mejores combinaciones y la puntuación en prueba del mejor modelo.

In [8]:
from sklearn.model_selection import GridSearchCV

rejilla = {
    "prepro__num__imputar__strategy": ["mean", "median"],
    "prepro__num__escalar": [StandardScaler(), "passthrough"],
    "modelo__alpha": [0.01, 0.1, 1, 10, 100],
}
busqueda = GridSearchCV(Pipeline([("prepro", crear_prepro()), ("modelo", Ridge())]), rejilla, cv=5,
                        scoring="neg_mean_absolute_error")
busqueda.fit(X_ent, y_ent)

resultados = pd.DataFrame(busqueda.cv_results_).sort_values("rank_test_score")
resultados["escalar"] = resultados["param_prepro__num__escalar"].map(lambda p: "no" if p == "passthrough" else "sí")
print(resultados[["param_prepro__num__imputar__strategy", "escalar", "param_modelo__alpha", "mean_test_score"]]
      .head().rename(columns=lambda c: c.removeprefix("param_")).to_string(index=False))
mae_rejilla = mean_absolute_error(y_pru, busqueda.predict(X_pru))
print(f"\nCombinaciones probadas: {len(resultados)}; MAE en prueba del mejor: {mae_rejilla:.2f} €")
assert len(resultados) == 2 * 2 * 5 and busqueda.best_score_ == resultados["mean_test_score"].max()

prepro__num__imputar__strategy escalar  modelo__alpha  mean_test_score
                          mean      no           1.00       -10.909487
                          mean      sí           1.00       -10.917419
                          mean      no           0.10       -10.924015
                          mean      sí           0.10       -10.924997
                          mean      no           0.01       -10.926480

Combinaciones probadas: 20; MAE en prueba del mejor: 9.89 €


## 9. `RandomizedSearchCV` con un modelo de árboles

Con muchos hiperparámetros continuos una rejilla crece demasiado. Usar `RandomizedSearchCV` para probar 20 combinaciones al azar de un `HistGradientBoostingRegressor` dentro del mismo pipeline, con la tasa de aprendizaje de una distribución log-uniforme y el número de hojas y la profundidad de distribuciones enteras de `scipy.stats`. Comparar el error en prueba con el de la rejilla de `Ridge`.

In [9]:
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

distribuciones = {
    "modelo__learning_rate": loguniform(0.01, 0.5),
    "modelo__max_leaf_nodes": randint(4, 32),
    "modelo__max_depth": randint(2, 6),
}
aleatoria = RandomizedSearchCV(
    Pipeline([("prepro", crear_prepro()), ("modelo", HistGradientBoostingRegressor(random_state=0))]),
    distribuciones, n_iter=20, cv=5, scoring="neg_mean_absolute_error", random_state=0)
aleatoria.fit(X_ent, y_ent)

print("Mejores parámetros:", {k.removeprefix("modelo__"): round(float(v), 3) if isinstance(v, float) else int(v) for k, v in aleatoria.best_params_.items()})
mae_aleatoria = mean_absolute_error(y_pru, aleatoria.predict(X_pru))
print(f"MAE en prueba: árboles {mae_aleatoria:.2f} € frente a Ridge {mae_rejilla:.2f} €")
assert len(aleatoria.cv_results_["params"]) == 20

Mejores parámetros: {'learning_rate': 0.092, 'max_depth': 3, 'max_leaf_nodes': 29}
MAE en prueba: árboles 13.82 € frente a Ridge 9.89 €


Los datos se generaron con una relación lineal, así que es razonable que `Ridge` gane a los árboles: un modelo más flexible no siempre es mejor.

## 10. `FunctionTransformer`

Añadir dos variables derivadas con `FunctionTransformer`: el logaritmo de la distancia a la playa (`np.log1p`, que conserva el nombre de la columna) y los metros cuadrados por habitación, calculados con una función propia que también da nombre a su columna. Comprobar los valores de una fila a mano y si el error del modelo cambia.

In [10]:
from sklearn.preprocessing import FunctionTransformer


def m2_por_habitacion(df):
    return (df["superficie"] / df["habitaciones"]).to_frame("m2_por_hab")


def nombre_m2(transformador, entradas):
    return np.array(["m2_por_hab"])


log_distancia = FunctionTransformer(np.log1p, feature_names_out="one-to-one")
ratio = Pipeline([("calcular", FunctionTransformer(m2_por_habitacion, feature_names_out=nombre_m2)),
                  ("imputar", SimpleImputer(strategy="median"))])
derivadas = ColumnTransformer([("log", log_distancia, ["distancia_playa"]),
                               ("ratio", ratio, ["superficie", "habitaciones"])]).set_output(transform="pandas")
D = derivadas.fit_transform(X_ent)
fila = X_ent.dropna(subset=["superficie"]).index[0]
print(D.loc[fila].round(3).to_dict())
assert np.isclose(D.loc[fila, "log__distancia_playa"], np.log1p(X_ent.loc[fila, "distancia_playa"]))
assert np.isclose(D.loc[fila, "ratio__m2_por_hab"], X_ent.loc[fila, "superficie"] / X_ent.loc[fila, "habitaciones"])

con_derivadas = Pipeline([
    ("prepro", ColumnTransformer([("base", crear_prepro(), X.columns.tolist()),
                                  ("derivadas", derivadas, X.columns.tolist())])),
    ("modelo", Ridge(alpha=1.0))]).fit(X_ent, y_ent)
print(f"MAE con variables derivadas: {mean_absolute_error(y_pru, con_derivadas.predict(X_pru)):.2f} € "
      f"(sin ellas: {mean_absolute_error(y_pru, pred):.2f} €)")

{'log__distancia_playa': 1.125, 'ratio__m2_por_hab': 27.0}
MAE con variables derivadas: 10.55 € (sin ellas: 9.98 €)


Aquí las variables derivadas no ayudan: el precio se generó lineal en la superficie y en la distancia, así que el logaritmo y el cociente solo añaden columnas correladas. Con datos reales conviene comprobarlo siempre con validación en lugar de suponerlo.

## 11. Un transformador propio con `fit` y `transform`

Escribir la clase `RecortadorPercentiles`, que en `fit` aprende los percentiles inferior y superior de cada columna y en `transform` recorta los valores a ese rango. Heredar de `BaseEstimator` y `TransformerMixin` para que tenga `get_params`, `fit_transform` y funcione con `clone` y `GridSearchCV`. Comprobar que avisa si se usa antes de entrenar, que los límites salen solo de los datos de entrenamiento y que se puede ajustar su parámetro en una búsqueda.

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted


class RecortadorPercentiles(TransformerMixin, BaseEstimator):
    def __init__(self, inferior=1.0, superior=99.0):
        self.inferior = inferior
        self.superior = superior

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.limites_ = np.nanpercentile(X, [self.inferior, self.superior], axis=0)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self)
        return np.clip(np.asarray(X, dtype=float), self.limites_[0], self.limites_[1])

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)


recortador = RecortadorPercentiles(5, 95)
try:
    recortador.transform([[1.0]])
except NotFittedError as e:
    print("Antes de fit:", type(e).__name__)

R = recortador.fit_transform(X_ent[["superficie", "distancia_playa"]])
print("Límites aprendidos (superficie, distancia):", recortador.limites_.round(2).tolist())
assert np.allclose(recortador.limites_[:, 1], np.nanpercentile(X_ent["distancia_playa"], [5, 95]))
assert np.nanmax(R[:, 1]) <= recortador.limites_[1, 1]
assert clone(recortador).get_params() == {"inferior": 5, "superior": 95}

con_recorte = Pipeline([
    ("prepro", ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("recortar", RecortadorPercentiles()),
                          ("escalar", StandardScaler())]), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["tipo"]),
        ("isla", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                           ("codificar", OneHotEncoder(handle_unknown="ignore"))]), ["isla"]),
        ("bin", "passthrough", binarias)])),
    ("modelo", Ridge())])
busqueda_recorte = GridSearchCV(con_recorte, {"prepro__num__recortar__superior": [90, 95, 99, 100]}, cv=5,
                                scoring="neg_mean_absolute_error").fit(X_ent, y_ent)
for p, s in zip(busqueda_recorte.cv_results_["params"], busqueda_recorte.cv_results_["mean_test_score"]):
    print(f"percentil superior {p['prepro__num__recortar__superior']:>3}: MAE validación {-s:.2f} €")

Antes de fit: NotFittedError
Límites aprendidos (superficie, distancia): [[30.0, 0.14], [111.15, 7.47]]


percentil superior  90: MAE validación 14.23 €
percentil superior  95: MAE validación 12.64 €
percentil superior  99: MAE validación 11.23 €
percentil superior 100: MAE validación 11.00 €


## 12. Selección de variables

Generar con `make_regression` un problema de 20 variables en el que solo 5 influyen de verdad (se conocen gracias a `coef=True`). Comparar las variables que eligen tres métodos: el filtro univariante `SelectKBest(f_regression)`, la eliminación recursiva `RFE` con una regresión lineal y `SelectFromModel` con un `LassoCV`, que descarta las variables cuyo coeficiente se anula. Comprobar cuáles aciertan el conjunto verdadero y por qué el filtro univariante, que mira cada variable por separado, puede perder una variable de efecto pequeño.

In [12]:
from sklearn.datasets import make_regression
from sklearn.feature_selection import RFE, SelectFromModel, f_regression
from sklearn.linear_model import LassoCV, LinearRegression

X_sel, y_sel, coef_real = make_regression(n_samples=300, n_features=20, n_informative=5, noise=15, coef=True,
                                          random_state=0)
verdaderas = {int(i) for i in np.flatnonzero(coef_real)}
selectores = {
    "SelectKBest": SelectKBest(f_regression, k=5),
    "RFE": RFE(LinearRegression(), n_features_to_select=5),
    "SelectFromModel(Lasso)": SelectFromModel(LassoCV(cv=5, random_state=0), threshold=1.0),
}
print("Verdaderas:", sorted(verdaderas), "con coeficientes", coef_real[sorted(verdaderas)].round(1).tolist())
aciertos = {}
for nombre, selector in selectores.items():
    elegidas = {int(i) for i in np.flatnonzero(selector.fit(X_sel, y_sel).get_support())}
    aciertos[nombre] = elegidas == verdaderas
    print(f"{nombre:24} {sorted(elegidas)}  {'acierta' if aciertos[nombre] else 'difiere'}")
assert aciertos["RFE"] and aciertos["SelectFromModel(Lasso)"]

Verdaderas: [3, 6, 12, 14, 18] con coeficientes [44.1, 11.9, 9.3, 84.4, 63.0]
SelectKBest              [3, 6, 10, 14, 18]  difiere
RFE                      [3, 6, 12, 14, 18]  acierta
SelectFromModel(Lasso)   [3, 6, 12, 14, 18]  acierta


## 13. Guardar y cargar el modelo con `joblib`

Guardar el mejor pipeline de la búsqueda en rejilla en `modelo_alquileres.joblib`, cargarlo de nuevo y comprobar que predice exactamente lo mismo. Como el pipeline incluye el preprocesado, el modelo cargado acepta directamente filas crudas: predecir el precio de dos alojamientos nuevos, uno de ellos en una isla que no estaba en los datos y otro con la valoración vacía.

In [13]:
import joblib

joblib.dump(busqueda.best_estimator_, "modelo_alquileres.joblib")
cargado = joblib.load("modelo_alquileres.joblib")
assert np.array_equal(cargado.predict(X_pru), busqueda.predict(X_pru))

nuevos = pd.DataFrame({
    "isla": ["La Graciosa", "Lanzarote"], "tipo": ["casa", "villa"], "superficie": [70, 160],
    "habitaciones": [2, 4], "distancia_playa": [0.3, 1.5], "piscina": [0, 1], "valoracion": [4.6, np.nan],
})
for (_, fila), precio_estimado in zip(nuevos.iterrows(), cargado.predict(nuevos)):
    print(f"{fila['tipo']:6} en {fila['isla']:12} {fila['superficie']:>4.0f} m²: {precio_estimado:7.2f} €/noche")
print("Pasos del modelo guardado:", list(cargado.named_steps))

casa   en La Graciosa    70 m²:  147.40 €/noche
villa  en Lanzarote     160 m²:  311.22 €/noche
Pasos del modelo guardado: ['prepro', 'modelo']
